In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Load Data 

In [2]:
data_path = Path("../data")

movie_path = f"{data_path}/refine_movies.csv"
rating_path = f"{data_path}/refine_ratings.csv"
tag_path = f"{data_path}/refine_tags.csv"
link_path = f"{data_path}/refine_links.csv"

In [3]:
movies_df = pd.read_csv(
    movie_path,
    usecols=['movieId', 'title','year','genres'],
    dtype={'title':str, 'year':str, 'movieId':int, 'genres':str})

ratings_df = pd.read_csv(
    rating_path,
    usecols=['userId','movieId','rating'],
    dtype={'userId':int, 'movieId':int, 'rating':float}
    )
tags_df = pd.read_csv(
    tag_path,
    usecols=['userId','movieId','tag'],
    dtype={'userId':int, 'movieId':int, 'tag':str})
links_df = pd.read_csv(link_path)

In [4]:
df_list = {
    "movies" : movies_df,
    "ratings" : ratings_df,
    "tags" : tags_df,
    "links_df" : links_df
}

# Data check

In [5]:
# genres to list
import ast
movies_df['genres'] = movies_df['genres'].apply(lambda x : ast.literal_eval(x))

movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9734                 [Action, Animation, Comedy, Fantasy]
9735                         [Animation, Comedy, Fantasy]
9736                                              [Drama]
9737                                  [Action, Animation]
9738                                             [Comedy]
Name: genres, Length: 9739, dtype: object

In [6]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [7]:
tags_df

,userId,movieId,tag
0,2,60756,['funny']
1,2,60756,"['Highly', 'quotable']"
2,2,60756,"['will', 'ferrell']"
3,2,89774,"['Boxing', 'story']"
4,2,89774,['MMA']
...,...,...,...
3678,606,7382,"['for', 'katie']"
3679,606,7936,['austere']
3680,610,3265,"['gun', 'fu']"
3681,610,3265,"['heroic', 'bloodshed']"


# Data Merge

## (1) rating centering
: 유저 bias 제거 . 분산도 함께 반영

### Z score normalization

In [8]:
temp_rating = ratings_df.copy()

func = lambda rating : (rating - rating.mean()) / rating.std()

temp_rating['centering_rating'] = ratings_df.groupby(by='userId')['rating'].transform(func) # apply 는 원래 구조를 변경, transform 는 원래 DF 구조를 유지

temp_rating['centering_rating'] = temp_rating['centering_rating'].fillna(0) # 평가가 1개인 경우 std=0 이라서 NaN 값이 출력 → 0 으로 수정

ratings_df = temp_rating

In [9]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movieId', how='left') # on : merge key
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


# zero ratings movie check

In [10]:
movie_ratings_df.isnull().sum()

userId              0
movieId             0
rating              0
centering_rating    0
title               0
genres              0
year                0
dtype: int64

In [11]:
no_ratings_movieId = movie_ratings_df.loc[movie_ratings_df.isnull().any(axis=1)]['movieId'].values
print("평가가 존재하지 않는 영화 ID 갯수: ",len(no_ratings_movieId))
no_ratings_movieId

평가가 존재하지 않는 영화 ID 갯수:  0


array([], dtype=int64)

In [12]:
movie_ratings_df[movie_ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centering_rating,title,genres,year


In [13]:
# double check (ratings_df 에도 존재하지 않는지 확인)
ratings_df[ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centering_rating


# Item User Matrix

In [14]:
item_user_matrix = movie_ratings_df.pivot_table(values='centering_rating', index='title', columns='userId')

In [15]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


## (1) 영화당 평균 평점 갯수 계산

In [16]:
print("평균 평점 갯수 :" ,round(item_user_matrix.apply(lambda x: x.count(), axis=1).mean()))
ratings_count = item_user_matrix.apply(lambda x: x.count(), axis=1)
ratings_count

평균 평점 갯수 : 10


title
'71 (2014)                                    1
'Hellboy': The Seeds of Creation (2004)       1
'Round Midnight (1986)                        2
'Salem's Lot (2004)                           1
'Til There Was You (1997)                     2
                                             ..
eXistenZ (1999)                              22
xXx (2002)                                   24
xXx: State of the Union (2005)                5
¡Three Amigos! (1986)                        26
À nous la liberté (Freedom for Us) (1931)     1
Length: 9721, dtype: int64

In [17]:
ratings_count.sort_values(ascending=False)

title
Forrest Gump (1994)                          329
Shawshank Redemption, The (1994)             317
Pulp Fiction (1994)                          307
Silence of the Lambs, The (1991)             279
Matrix, The (1999)                           278
                                            ... 
King Solomon's Mines (1937)                    1
King Ralph (1991)                              1
King Kong Lives (1986)                         1
Kindred, The (1986)                            1
À nous la liberté (Freedom for Us) (1931)      1
Length: 9721, dtype: int64

In [18]:
movies_df

,movieId,title,genres,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995)
4,5,Father of the Bride Part II (1995),[Comedy],(1995)
...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017)
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017)
9736,193585,Flint (2017),[Drama],(2017)
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018)


In [19]:
temp_df = movies_df.set_index(keys='title').copy()
temp_df = temp_df.join(ratings_count.rename('count'))
movies_df = temp_df.reset_index()

movies_df['count'] = movies_df['count']
movies_df['count'] = movies_df['count'].fillna(0)
movies_df['count'] = movies_df['count'].astype('int64')

movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [20]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


# 유저별 장르 평균 점수

In [21]:
genres_list = movies_df['genres'].explode().unique()

In [22]:
movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [23]:
def calc_user_genre_mean(group):
    genres_totals = {genre : 0 for genre in genres_list}
    genres_count = {genre : 0 for genre in genres_list}
    
    for _, row in group.iterrows():
        rating = row['centering_rating']
        movie_genres = row['genres']
        for genre in movie_genres:
            genres_totals[genre] += rating
            genres_count[genre] += 1
    genres_avg = {}
    for genre in genres_list:
        if genres_count[genre] > 0 :
            genres_avg[genre] = genres_totals[genre]/genres_count[genre]
        else:
            genres_avg[genre] = pd.NA
    return pd.Series(genres_avg)

In [24]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


In [25]:
user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_45928/2273720666.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)


In [26]:
user_genre_avg

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
userId,,,,,,,,,,,,,,,,,,,
1,0.027318,0.404071,0.226536,-0.111582,-0.085629,-0.073354,0.203778,-0.055193,-0.013529,-0.276139,-1.119672,-0.249626,-0.176714,0.167016,0.394275,<NA>,<NA>,-0.100825,0.791978
2,0.271086,<NA>,<NA>,0.064205,<NA>,0.684849,-0.081829,0.007782,-0.184053,-0.308182,-1.177084,0.064205,-0.090956,0.684849,<NA>,0.477967,-0.246118,-0.55644,<NA>
3,0.030662,-0.925982,-0.925982,-0.686821,0.449193,-0.925982,-0.806402,0.54315,-0.925982,0.816476,1.076991,1.226467,0.843809,-0.925982,-0.925982,<NA>,<NA>,<NA>,<NA>
4,0.0758,0.338185,0.186002,-0.034957,0.097896,-0.134108,-0.054955,-0.179238,0.197275,-0.002225,0.528415,-0.058815,-0.549551,0.012078,0.338185,0.338185,-0.422732,0.186002,0.338185
5,-0.390093,0.703697,0.47933,-0.171335,0.511382,-0.550719,0.165216,-0.530322,0.198871,-0.081588,-0.642506,0.367146,-1.147331,-0.305955,0.771007,<NA>,0.030596,-0.642506,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.212669,0.07856,-0.287824,-0.127159,-0.082115,0.114303,0.180310,-0.660928,-0.004507,-0.182668,-0.429825,0.184789,-0.138702,0.186307,0.096494,0.19693,-0.821547,-0.339218,0.214192
607,-0.33079,-0.468865,-0.378026,-0.475141,-0.222302,-0.278416,0.234140,-0.066146,0.02974,0.340346,0.339861,0.891582,-0.555162,0.394105,-0.192715,<NA>,1.257075,0.221511,<NA>
608,0.080443,-0.014819,-0.624453,-0.368359,-0.124322,-0.229215,0.281048,0.181744,0.443672,0.372944,0.171795,0.385957,0.150317,0.412107,-0.348942,-0.124322,0.802237,-0.461252,0.570598


# 영화별 장르 매트릭스 계산

In [27]:
genres_list

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [28]:
movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [29]:
movie_genre_matrix = pd.DataFrame(0, index=movies_df['title'], columns=genres_list)
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Waiting to Exhale (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Father of the Bride Part II (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
No Game No Life: Zero (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Flint (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [30]:
for genre in genres_list:
    movie_genre_matrix[genre] = movies_df.set_index('title')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Waiting to Exhale (1995),0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
Father of the Bride Part II (1995),0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
No Game No Life: Zero (2017),0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Flint (2017),0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [31]:
genre_count = movie_genre_matrix.sum(axis=1)

movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0) # 장르가 많은 영화의 점수가 많이 반영되지 않도록 정규화

movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Jumanji (1995),0.333333,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Grumpier Old Men (1995),0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Waiting to Exhale (1995),0.000000,0.000000,0.000000,0.333333,0.000000,0.333333,0.333333,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Father of the Bride Part II (1995),0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
No Game No Life: Zero (2017),0.000000,0.333333,0.000000,0.333333,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Flint (2017),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
print(user_genre_avg.shape)
print(movie_genre_matrix.shape)

(610, 19)
(9739, 19)


# 예측 평점 행렬 계산

user_genre_avg 와 movie_genre_matrix 내적
- 결과 shape 사용자 수 x 영화 수

In [33]:
pred_ratings_use_genre = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)
pred_ratings_use_genre

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_45928/2084021829.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pred_ratings_use_genre = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)


title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.056075,-0.092468,0.006281,-0.111582,-0.114954,-0.092468,0.126927,-0.055193,-0.101338,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,-0.111582
2,0.067058,0.090362,0.374527,0.222408,0.064205,-0.161484,0.374527,0.135543,0.007782,-0.009771,...,-0.004742,-0.040915,-0.008812,0.000000,0.477967,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-0.411786,-0.148709,-0.806402,-0.806402,-0.686821,0.144548,-0.806402,-0.447660,0.543150,0.463429,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-0.686821
4,0.132585,0.119899,-0.084532,-0.074673,-0.034957,0.005271,-0.084532,0.130901,-0.179238,-0.035221,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.034957
5,0.226596,0.200206,-0.361027,-0.185613,-0.171335,-0.137680,-0.361027,0.044618,-0.530322,-0.334001,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,-0.171335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.126241,-0.194203,-0.006428,0.055818,-0.127159,-0.282701,-0.006428,-0.250247,-0.660928,-0.352088,...,-0.212057,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.127159
607,-0.375025,-0.310373,-0.376779,-0.173139,-0.475141,0.101314,-0.376779,-0.354408,-0.066146,-0.018863,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,-0.475141
608,-0.210302,-0.222777,-0.298787,-0.105509,-0.368359,0.332786,-0.298787,-0.272005,0.181744,0.211710,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-0.368359


# fill NaN value in movie-user matrix

## (1) sim matrix

In [34]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


In [35]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_fill_zero = cosine_similarity(item_user_matrix.fillna(0)) # 평가가 유사한것이 유사도의 기준 

cos_sim_zero_df = pd.DataFrame(cos_sim_fill_zero, index=item_user_matrix.index, columns=item_user_matrix.index)
cos_sim_zero_df.head(5)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,-0.033186,0.0,...,0.0,0.127241,-0.223349,-0.753563,0.0,0.0,-0.35184,-0.615944,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,-0.298277,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,-0.298277,1.000000,0.000000,0.000000,0.0,0.089156,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.947758,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.947758,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0


# (2) genre 이용 sim matrix

In [36]:
pred_ratings_use_genre.T[pred_ratings_use_genre.T.index=="'71 (2014)"]

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.009865,0.075655,-0.09319,-0.056085,-0.188162,0.110677,0.03577,0.062119,-0.087317,0.077702,...,0.036036,0.250859,-0.037388,-0.047362,-0.244458,-0.119244,0.225611,0.311961,0.091045,0.020766


In [37]:
pred_ratings_use_genre.T

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0.092143,0.067058,-0.411786,0.132585,0.226596,0.240394,-0.002281,0.284153,0.654182,0.232814,...,0.137212,0.052801,-0.389747,-0.017829,-0.055557,-0.126241,-0.375025,-0.210302,-0.384535,0.032743
Jumanji (1995),0.056075,0.090362,-0.148709,0.119899,0.200206,0.222657,-0.027767,0.110405,0.790394,0.224901,...,0.177435,-0.087448,-0.227352,0.100047,-0.002992,-0.194203,-0.310373,-0.222777,-0.452226,-0.044776
Grumpier Old Men (1995),-0.092468,0.374527,-0.806402,-0.084532,-0.361027,-0.001703,-0.243406,-0.226571,0.122431,0.017867,...,-0.139026,-0.125339,-0.113648,-0.434308,0.102673,-0.006428,-0.376779,-0.298787,-0.060888,0.049639
Waiting to Exhale (1995),0.006281,0.222408,-0.806402,-0.074673,-0.185613,0.046145,-0.187011,-0.077339,0.125550,-0.023753,...,-0.081375,-0.117119,-0.046285,-0.286326,0.031088,0.055818,-0.173139,-0.105509,0.032076,0.105474
Father of the Bride Part II (1995),-0.111582,0.064205,-0.686821,-0.034957,-0.171335,-0.145244,-0.050390,-0.376555,0.318891,-0.010843,...,-0.166505,-0.094667,-0.176258,-0.320260,0.058652,-0.127159,-0.475141,-0.368359,0.034303,0.049669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0.037917,0.017997,-0.155115,0.055472,0.128355,0.180692,-0.007878,0.126962,0.539908,0.204002,...,0.060255,0.259522,-0.378701,-0.069919,-0.080639,-0.197910,-0.308114,-0.081439,-0.391170,0.020914
No Game No Life: Zero (2017),0.068953,0.021402,-0.387870,0.133708,0.347915,0.195568,-0.017411,0.251948,0.755468,0.209223,...,0.124949,0.285082,-0.409156,-0.158095,-0.113312,-0.043571,-0.388769,-0.169167,-0.388766,0.062088
Flint (2017),0.203778,-0.081829,-0.806402,-0.054955,0.165216,0.141839,-0.074221,0.221125,0.131787,-0.106995,...,0.033928,-0.100681,0.088440,0.009636,-0.112081,0.180310,0.234140,0.281048,0.218004,0.217143


# (3) Year

In [38]:
movies_year = movies_df['year'].str.strip(to_strip="()")

## 연도 추가 전처리

In [39]:
movies_year[movies_year.apply(lambda x : len(x) > 4)]
movies_df.loc[9515, 'year'] = "(2006)"
movies_df.loc[9515]

title          Death Note: Desu nôto (2006–2007)
movieId                                   171749
genres     [Animation, Mystery, Sci-Fi, Fantasy]
year                                      (2006)
count                                          1
Name: 9515, dtype: object

In [40]:
movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


## 연도의 괄호 제거

In [41]:
movies_year = movies_df.set_index('title')['year'].str.strip(to_strip="()")

In [42]:
movies_year

title
Toy Story (1995)                             1995
Jumanji (1995)                               1995
Grumpier Old Men (1995)                      1995
Waiting to Exhale (1995)                     1995
Father of the Bride Part II (1995)           1995
                                             ... 
Black Butler: Book of the Atlantic (2017)    2017
No Game No Life: Zero (2017)                 2017
Flint (2017)                                 2017
Bungo Stray Dogs: Dead Apple (2018)          2018
Andrew Dice Clay: Dice Rules (1991)          1991
Name: year, Length: 9739, dtype: object

In [43]:
movie_ratings_df['year'] = movie_ratings_df['title'].map(movies_year.to_dict())

movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017


### 연도가 조사되지 않은 영화 확인
- 전처리 체크

In [44]:
print("연도 영화하지 않는 영화 : " ,movie_ratings_df['year'].isnull().sum(), ", 전체 :", len(movie_ratings_df))

연도 영화하지 않는 영화 :  0 , 전체 : 100836


## 각 연도별 평균 부여 점수 계산

In [45]:
movie_ratings_df.groupby(['userId','year'])['rating'].mean()

userId  year
1       1922    4.000000
        1930    5.000000
        1931    4.000000
        1933    4.500000
        1938    5.000000
                  ...   
610     2013    3.626866
        2014    3.791667
        2015    3.632653
        2016    3.741935
        2017    4.400000
Name: rating, Length: 17440, dtype: float64

# 5년 단위 연도 합산
- 1년단위 진행시 sparse matrix

In [46]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017


In [47]:
user_year_matrix = movie_ratings_df.groupby(['userId','year'])['rating'].mean().unstack()

user_year_matrix

year,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.750000,4.083333,3.750000,3.500000,5.000000,3.000000,4.500000,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,...,3.500000,4.000000,NaN,3.375000,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 영화에 대한 matrix 계산

In [48]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017


In [49]:
movie_year_matrix = pd.get_dummies(movies_year).astype('int64')

movie_year_matrix

,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Waiting to Exhale (1995),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Father of the Bride Part II (1995),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
No Game No Life: Zero (2017),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
Flint (2017),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [50]:
user_year_matrix.count(axis=1)

userId
1      53
2      15
3      23
4      47
5      10
       ..
606    85
607    40
608    50
609     9
610    63
Length: 610, dtype: int64

In [51]:
print(movie_year_matrix.shape)
print(user_year_matrix.shape)

(9739, 106)
(610, 106)


In [52]:
user_year_matrix = user_year_matrix.apply(lambda x : (x - x.mean())/x.std())
user_year_matrix

year,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.597425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.003776,0.351061,0.109923,-0.336696,1.854890,-1.098558,1.024749,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-4.137905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.21356,NaN,...,-0.321786,0.237316,NaN,-0.498609,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
pred_ratings_use_year = user_year_matrix.fillna(0).dot(movie_year_matrix.T)
pred_ratings_use_year

title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
userId,,,,,,,,,,,,,,,,,,,,,
1,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.655538
2,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,...,0.351061,1.854890,-1.098558,1.024749,1.024749,0.00000,0.00000,0.00000,0.0,0.000000
3,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-3.829297
4,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-0.514419
5,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.265552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,...,0.237316,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-0.074113
607,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.499543
608,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-1.799078


In [54]:
pred_ratings_use_genre

title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.056075,-0.092468,0.006281,-0.111582,-0.114954,-0.092468,0.126927,-0.055193,-0.101338,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,-0.111582
2,0.067058,0.090362,0.374527,0.222408,0.064205,-0.161484,0.374527,0.135543,0.007782,-0.009771,...,-0.004742,-0.040915,-0.008812,0.000000,0.477967,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-0.411786,-0.148709,-0.806402,-0.806402,-0.686821,0.144548,-0.806402,-0.447660,0.543150,0.463429,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-0.686821
4,0.132585,0.119899,-0.084532,-0.074673,-0.034957,0.005271,-0.084532,0.130901,-0.179238,-0.035221,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.034957
5,0.226596,0.200206,-0.361027,-0.185613,-0.171335,-0.137680,-0.361027,0.044618,-0.530322,-0.334001,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,-0.171335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.126241,-0.194203,-0.006428,0.055818,-0.127159,-0.282701,-0.006428,-0.250247,-0.660928,-0.352088,...,-0.212057,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.127159
607,-0.375025,-0.310373,-0.376779,-0.173139,-0.475141,0.101314,-0.376779,-0.354408,-0.066146,-0.018863,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,-0.475141
608,-0.210302,-0.222777,-0.298787,-0.105509,-0.368359,0.332786,-0.298787,-0.272005,0.181744,0.211710,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-0.368359


In [ ]:
pred_ratings_use_genre_year = pred_ratings_use_genre.add(pred_ratings_use_year)
pred_ratings_use_genre_year

title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
userId,,,,,,,,,,,,,,,,,,,,,
1,1.457062,1.420994,1.272451,1.371200,1.253337,1.249965,1.272451,1.491846,1.309726,1.263581,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,0.543956
2,0.579774,0.603077,0.887242,0.735123,0.576920,0.351231,0.887242,0.648258,0.520498,0.502944,...,0.346319,1.813975,-1.107371,1.024749,1.502717,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-4.586191,-4.323114,-4.980807,-4.980807,-4.861226,-4.029857,-4.980807,-4.622065,-3.631255,-3.710975,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-4.516119
4,-0.827794,-0.840480,-1.044912,-1.035053,-0.995336,-0.955109,-1.044912,-0.829478,-1.139618,-0.995600,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.549376
5,0.292919,0.266529,-0.294704,-0.119290,-0.105012,-0.071357,-0.294704,0.110942,-0.463999,-0.267678,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,0.094217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.132457,-0.200418,-0.012643,0.049603,-0.133375,-0.288917,-0.012643,-0.256462,-0.667143,-0.358304,...,0.025259,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.201272
607,-0.197104,-0.132452,-0.198858,0.004782,-0.297220,0.279235,-0.198858,-0.176487,0.111776,0.159058,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,0.024402
608,-1.364435,-1.376910,-1.452920,-1.259642,-1.522492,-0.821346,-1.452920,-1.426138,-0.972389,-0.942423,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-2.167437


In [64]:
cos_sim_genre_year = cosine_similarity(pred_ratings_use_genre_year.T)

sim_matrix = pd.DataFrame(cos_sim_genre_year, movies_df['title'], movies_df['title'])

In [66]:
sim_matrix

title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1.000000,0.992201,0.937751,0.939113,0.942758,0.900123,0.937751,0.982575,0.893879,0.933102,...,0.250952,0.284705,0.216505,0.264727,0.082266,0.204666,0.243542,0.042353,0.194310,0.371159
Jumanji (1995),0.992201,1.000000,0.917556,0.920493,0.920617,0.889105,0.917556,0.985755,0.889646,0.930949,...,0.242261,0.254766,0.199724,0.222961,0.076510,0.198745,0.232152,0.031141,0.164454,0.348095
Grumpier Old Men (1995),0.937751,0.917556,1.000000,0.991114,0.976337,0.905018,1.000000,0.908562,0.889149,0.918604,...,0.200273,0.185155,0.255940,0.084693,0.067052,0.063976,0.089834,0.071636,0.005915,0.375759
Waiting to Exhale (1995),0.939113,0.920493,0.991114,1.000000,0.965673,0.930860,0.991114,0.911524,0.900818,0.934912,...,0.178628,0.198690,0.260956,0.071543,0.074925,0.032585,0.055232,0.140303,-0.015937,0.354718
Father of the Bride Part II (1995),0.942758,0.920617,0.976337,0.965673,1.000000,0.900568,0.976337,0.909629,0.887640,0.914038,...,0.219389,0.165997,0.270785,0.090914,0.068291,0.100089,0.127486,0.027023,0.023113,0.403320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0.204666,0.198745,0.063976,0.032585,0.100089,0.038372,0.063976,0.175210,0.110720,0.082147,...,0.405170,0.420108,0.207357,0.592009,0.252292,1.000000,0.965138,0.453875,0.610263,0.114300
No Game No Life: Zero (2017),0.243542,0.232152,0.089834,0.055232,0.127486,0.003399,0.089834,0.197103,0.039002,0.046048,...,0.362006,0.462037,0.221197,0.648677,0.224709,0.965138,1.000000,0.408236,0.576274,0.138270
Flint (2017),0.042353,0.031141,0.071636,0.140303,0.027023,0.114799,0.071636,0.034465,0.051231,0.067910,...,0.112636,0.328660,0.320669,0.152490,0.264391,0.453875,0.408236,1.000000,0.023796,-0.038360


In [68]:
sim_matrix.iloc[0].sort_values(ascending=False)

title
Toy Story (1995)                        1.000000
Jumanji (1995)                          0.992201
Indian in the Cupboard, The (1995)      0.992201
Gordy (1995)                            0.991192
Kid in King Arthur's Court, A (1995)    0.990331
                                          ...   
Crossfire (1947)                       -0.082698
Lady from Shanghai, The (1947)         -0.084040
Corbeau, Le (Raven, The) (1943)        -0.093570
Shadow of a Doubt (1943)               -0.093570
Out of the Past (1947)                 -0.098503
Name: Toy Story (1995), Length: 9739, dtype: float64

# (4) Popular movies